In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositif utilisé : {device}")
if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU totale : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

Dispositif utilisé : cuda
GPU détecté : NVIDIA GeForce RTX 5070 Laptop GPU
Mémoire GPU totale : 8.1 Go


In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Jeu d'entraînement : {len(train_dataset)} exemples")
print(f"Jeu de test        : {len(test_dataset)} exemples")

100.0%
100.0%
100.0%
100.0%

Jeu d'entraînement : 60000 exemples
Jeu de test        : 10000 exemples


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1   = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2   = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1     = nn.Linear(64 * 7 * 7, 128)
        self.fc2     = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x.view(x.size(0), -1))
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = CNN().to(device)
print(f"Paramètres du modèle : {sum(p.numel() for p in model.parameters()):,}")

Paramètres du modèle : 421,642


In [4]:
import torch.optim as optim
import time

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

start_total = time.time()

for epoch in range(5):
    model.train()
    running_loss = 0.0
    start_epoch  = time.time()

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = criterion(model(data), target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    elapsed = time.time() - start_epoch
    print(f"Époque {epoch+1}/5 — perte : {running_loss/len(train_loader):.4f} — durée : {elapsed:.1f}s")

total = time.time() - start_total
print(f"\nEntraînement terminé en {total:.1f}s")

Époque 1/5 — perte : 0.1406 — durée : 7.4s
Époque 2/5 — perte : 0.0504 — durée : 6.4s
Époque 3/5 — perte : 0.0375 — durée : 6.3s
Époque 4/5 — perte : 0.0292 — durée : 6.2s
Époque 5/5 — perte : 0.0244 — durée : 6.4s

Entraînement terminé en 32.7s


In [5]:
model.eval()
correct = total_samples = 0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        pred = model(data).argmax(dim=1)
        correct       += pred.eq(target).sum().item()
        total_samples += target.size(0)

print(f"Précision sur le jeu de test : {100 * correct / total_samples:.2f}%")

Précision sur le jeu de test : 99.12%
